# Sistema de Recuperación de Información - SciFact

**Corpus asignado:** SciFact (BeIR)  
**Objetivo:** Indexar documentos científicos, recuperar resultados con TF-IDF + similitud coseno y BM25, evaluar con P@10, R@10 y NDCG@10 sobre los qrels de test, y exponer la búsqueda como servicio web.  

Este notebook implementa la parte de indexación, recuperación y evaluación **sin usar embeddings**. Toda la búsqueda es por términos.

## 1. Instalación de dependencias y recursos NLTK

Se requieren los siguientes paquetes:  
`datasets`, `scikit-learn`, `nltk`, `numpy`, `scipy`, `pandas`, `fastapi`, `uvicorn`, `pyngrok`  

```bash
pip install datasets scikit-learn nltk numpy scipy pandas fastapi uvicorn pyngrok
```

In [21]:
import re
import math
import json
from collections import defaultdict

import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

print('Dependencias cargadas correctamente')

Dependencias cargadas correctamente


## 2. Carga de SciFact y verificación de IDs

Se cargan:  
- **Corpus:** documentos con `_id`, `title`, `text`.  
- **Queries:** consultas con `_id`, `text`.  
- **Qrels:** juicios de relevancia de test con `query-id`, `corpus-id`, `score`.  

Se verifica que cada `query-id` y `corpus-id` de los qrels exista en los conjuntos correspondientes.

In [22]:
# Cargar datasets
corpus = load_dataset('BeIR/scifact', 'corpus', split='corpus')
queries = load_dataset('BeIR/scifact', 'queries', split='queries')
qrels = load_dataset('BeIR/scifact-qrels', split='test')

print('Corpus:', corpus)
print('Queries:', queries)
print('Qrels test:', qrels)
print()
print('Ejemplo corpus:', corpus[0])
print('Ejemplo query:', queries[0])
print('Ejemplo qrel:', qrels[0])

Corpus: Dataset({
    features: ['_id', 'title', 'text'],
    num_rows: 5183
})
Queries: Dataset({
    features: ['_id', 'title', 'text'],
    num_rows: 1109
})
Qrels test: Dataset({
    features: ['query-id', 'corpus-id', 'score'],
    num_rows: 339
})

Ejemplo corpus: {'_id': '4983', 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.', 'text': 'Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, earl

In [23]:
# Verificar correspondencia de IDs
corpus_ids = set(doc['_id'] for doc in corpus)
query_ids = set(q['_id'] for q in queries)
qrel_corpus_ids = set(str(q['corpus-id']) for q in qrels)
qrel_query_ids = set(str(q['query-id']) for q in qrels)

print(f'Total documentos en corpus: {len(corpus_ids)}')
print(f'Total consultas: {len(query_ids)}')
print(f'Total qrels de test: {len(qrels)}')
print(f'Queries distintas en qrels: {len(qrel_query_ids)}')
print(f'Documentos de qrels ausentes en corpus: {len(qrel_corpus_ids - corpus_ids)}')
print(f'Queries de qrels ausentes en queries: {len(qrel_query_ids - query_ids)}')
assert qrel_corpus_ids.issubset(corpus_ids), 'Hay corpus-id en qrels que no existen en el corpus'
assert qrel_query_ids.issubset(query_ids), 'Hay query-id en qrels que no existen en queries'
print('\n Todos los IDs de los qrels corresponden a documentos y consultas válidas.')

Total documentos en corpus: 5183
Total consultas: 1109
Total qrels de test: 339
Queries distintas en qrels: 300
Documentos de qrels ausentes en corpus: 0
Queries de qrels ausentes en queries: 0

 Todos los IDs de los qrels corresponden a documentos y consultas válidas.


## 3. Preprocesamiento e índice invertido

**Decisiones de preprocesamiento:**  
1. **Normalización:** conversión a minúsculas y eliminación de caracteres no alfanuméricos.  
2. **Tokenización:** `word_tokenize` de NLTK.  
3. **Stopwords:** lista en inglés de NLTK.  
4. **Stemming:** algoritmo de Porter.  
5. **Filtrado:** se conservan solo tokens alfabéticos de más de 2 caracteres.  

Cada documento se indexa como concatenación de `title + ' ' + text`, conservando su `_id` original.

In [24]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess(text):
    """Normaliza, tokeniza, elimina stopwords y aplica stemming."""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    tokens = word_tokenize(text)
    tokens = [
        stemmer.stem(tok)
        for tok in tokens
        if tok.isalpha() and tok not in stop_words and len(tok) > 2
    ]
    return tokens

# Preparar documentos conservando IDs originales
doc_ids = []
doc_titles = {}
doc_texts = {}
tokenized_docs = []

for doc in corpus:
    did = doc['_id']
    title = doc['title'] or ''
    text = doc['text'] or ''
    full = (title + ' ' + text).strip()
    tokens = preprocess(full)
    doc_ids.append(did)
    doc_titles[did] = title
    doc_texts[did] = text
    tokenized_docs.append(tokens)

print(f'Documentos indexados: {len(doc_ids)}')
print(f'Ejemplo de tokens (doc {doc_ids[0]}): {tokenized_docs[0][:30]}')

Documentos indexados: 5183
Ejemplo de tokens (doc 4983): ['microstructur', 'develop', 'human', 'newborn', 'cerebr', 'white', 'matter', 'assess', 'vivo', 'diffus', 'tensor', 'magnet', 'reson', 'imag', 'alter', 'architectur', 'cerebr', 'white', 'matter', 'develop', 'human', 'brain', 'affect', 'cortic', 'develop', 'result', 'function', 'disabl', 'line', 'scan']


In [25]:
# Construir índice invertido: término -> lista de índices de documentos
inverted_index = defaultdict(list)
for idx, tokens in enumerate(tokenized_docs):
    for term in set(tokens):
        inverted_index[term].append(idx)

vocab_size = len(inverted_index)
print(f'Tamaño del vocabulario: {vocab_size}')
print(f'Ejemplo de índice invertido: {list(inverted_index.keys())[0]} -> {inverted_index[list(inverted_index.keys())[0]][:5]}')

Tamaño del vocabulario: 19856
Ejemplo de índice invertido: effect -> [0, 5, 7, 8, 10]


## 4. Modelo TF-IDF con similitud coseno

Se entrena un `TfidfVectorizer` de scikit-learn sobre los documentos preprocesados ya unidos como cadenas. Para consultar, se preprocesa la consulta de la misma forma y se calcula la similitud coseno con todos los documentos.

In [26]:
# Entrenar TF-IDF con los documentos preprocesados (como cadenas)
preprocessed_docs = [' '.join(tokens) for tokens in tokenized_docs]
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(preprocessed_docs)

print(f'Matriz TF-IDF: {tfidf_matrix.shape}')

def search_tfidf(query, top_k=10):
    """Devuelve los top_k documentos más similares a la consulta usando TF-IDF + coseno."""
    q_tokens = preprocess(query)
    q_text = ' '.join(q_tokens)
    q_vec = tfidf_vectorizer.transform([q_text])
    similarities = cosine_similarity(q_vec, tfidf_matrix).flatten()
    top_idx = np.argsort(similarities)[::-1][:top_k]
    results = []
    for rank, idx in enumerate(top_idx, start=1):
        score = float(similarities[idx])
        if score <= 0:
            break
        did = doc_ids[idx]
        results.append((did, score))
    return results

# Prueba rápida
sample_query = next(q for q in queries if q['_id'] == '1')['text']
print('Consulta de prueba:', sample_query)
print('Top 3 TF-IDF:', search_tfidf(sample_query, 3))

Matriz TF-IDF: (5183, 19855)
Consulta de prueba: 0-dimensional biomaterials show inductive properties.
Top 3 TF-IDF: [('10906636', 0.10632252789889068), ('10608397', 0.10197873840324424), ('31543713', 0.09677198724941677)]


## 5. Modelo BM25

Implementación manual sobre el mismo índice invertido. Parámetros estándar: `k1=1.5`, `b=0.75`. La fórmula usada es el score BM25 estándar:

```
score(q, d) = sum_t in q [ idf(t) * (f(t,d) * (k1 + 1)) / (f(t,d) + k1 * (1 - b + b * |d| / avgdl)) ]
idf(t) = log((N - n_t + 0.5) / (n_t + 0.5))
```

In [27]:
# Estadísticas necesarias para BM25
N = len(tokenized_docs)
doc_lengths = np.array([len(tokens) for tokens in tokenized_docs], dtype=np.float64)
avgdl = float(np.mean(doc_lengths))

# Frecuencia de documentos por término
df = {term: len(docs) for term, docs in inverted_index.items()}

# IDF de BM25
idf = {
    term: math.log((N - n + 0.5) / (n + 0.5) + 1e-10)
    for term, n in df.items()
}

k1 = 1.5
b = 0.75

def search_bm25(query, top_k=10):
    """Devuelve los top_k documentos usando BM25 sobre el índice invertido."""
    q_tokens = preprocess(query)
    scores = np.zeros(N, dtype=np.float64)
    for term in q_tokens:
        if term not in inverted_index:
            continue
        postings = inverted_index[term]
        idf_term = idf[term]
        for doc_idx in postings:
            f = tokenized_docs[doc_idx].count(term)
            if f == 0:
                continue
            denom = f + k1 * (1 - b + b * doc_lengths[doc_idx] / avgdl)
            scores[doc_idx] += idf_term * f * (k1 + 1) / denom
    top_idx = np.argsort(scores)[::-1][:top_k]
    results = []
    for idx in top_idx:
        score = float(scores[idx])
        if score <= 0:
            break
        results.append((doc_ids[idx], score))
    return results

print('Top 3 BM25:', search_bm25(sample_query, 3))

Top 3 BM25: [('21257564', 9.749843781504081), ('18953920', 9.534339229867992), ('40212412', 9.224695024001619)]


## 6. Funciones de evaluación

Se implementan Precision@k, Recall@k y NDCG@k usando los qrels de test. NDCG se calcula manualmente con la fórmula estándar usando ganancias binarias.

In [28]:
# Convertir qrels a diccionario: query-id -> set de corpus-id relevantes
qrels_dict = defaultdict(set)
for qrel in qrels:
    qid = str(qrel['query-id'])
    cid = str(qrel['corpus-id'])
    if int(qrel['score']) > 0:
        qrels_dict[qid].add(cid)

def precision_at_k(retrieved, relevant, k=10):
    retrieved_k = retrieved[:k]
    if not retrieved_k:
        return 0.0
    return len(set(retrieved_k) & relevant) / k

def recall_at_k(retrieved, relevant, k=10):
    if not relevant:
        return 0.0
    return len(set(retrieved[:k]) & relevant) / len(relevant)

def dcg(relevances):
    return sum((2 ** rel - 1) / math.log2(i + 2) for i, rel in enumerate(relevances))

def ndcg_at_k(retrieved, relevant, k=10):
    rel = [1 if doc_id in relevant else 0 for doc_id in retrieved[:k]]
    ideal = sorted([1] * len(relevant) + [0] * (k - len(relevant)), reverse=True)[:k]
    ideal_dcg = dcg(ideal)
    if ideal_dcg == 0:
        return 0.0
    return dcg(rel) / ideal_dcg

assert math.isclose(precision_at_k(['a', 'b', 'c', 'd'], {'a'}, k=10), 0.10)
assert precision_at_k([], {'a'}, k=10) == 0.0
print('P@10 con 4 recuperados y 1 relevante:', precision_at_k(['a', 'b', 'c', 'd'], {'a'}, k=10))
print('P@10 con lista vacía:', precision_at_k([], {'a'}, k=10))

print(f'Queries de test con qrels: {len(qrels_dict)}')
print(f'Ejemplo qrels (query 1): {qrels_dict["1"]}')

P@10 con 4 recuperados y 1 relevante: 0.1
P@10 con lista vacía: 0.0
Queries de test con qrels: 300
Ejemplo qrels (query 1): {'31715818'}


## 7. Evaluación de TF-IDF y BM25 sobre test

Se evalúan ambos modelos con las 300 consultas de test que tienen qrels. Se reportan Precision@10, Recall@10 y NDCG@10 promedios.

In [29]:
# Diccionario de consultas por id
queries_dict = {q['_id']: q['text'] for q in queries}

# Listas para métricas
tfidf_p10, tfidf_r10, tfidf_ndcg10 = [], [], []
bm25_p10, bm25_r10, bm25_ndcg10 = [], [], []

for qid in sorted(qrels_dict.keys(), key=int):
    query_text = queries_dict[qid]
    relevant = qrels_dict[qid]

    tfidf_results = search_tfidf(query_text, top_k=10)
    tfidf_retrieved = [did for did, _ in tfidf_results]
    tfidf_p10.append(precision_at_k(tfidf_retrieved, relevant, k=10))
    tfidf_r10.append(recall_at_k(tfidf_retrieved, relevant, k=10))
    tfidf_ndcg10.append(ndcg_at_k(tfidf_retrieved, relevant, k=10))

    bm25_results = search_bm25(query_text, top_k=10)
    bm25_retrieved = [did for did, _ in bm25_results]
    bm25_p10.append(precision_at_k(bm25_retrieved, relevant, k=10))
    bm25_r10.append(recall_at_k(bm25_retrieved, relevant, k=10))
    bm25_ndcg10.append(ndcg_at_k(bm25_retrieved, relevant, k=10))

# Promedios
metrics = {
    'TF-IDF': {
        'Precision@10': np.mean(tfidf_p10),
        'Recall@10': np.mean(tfidf_r10),
        'NDCG@10': np.mean(tfidf_ndcg10),
    },
    'BM25': {
        'Precision@10': np.mean(bm25_p10),
        'Recall@10': np.mean(bm25_r10),
        'NDCG@10': np.mean(bm25_ndcg10),
    }
}

df_metrics = pd.DataFrame(metrics).T
print(df_metrics)

# Guardar resultados numéricos para referencia
print('\nResumen numérico:')
print(json.dumps({k: {m: round(v, 4) for m, v in vals.items()} for k, vals in metrics.items()}, indent=2))

        Precision@10  Recall@10   NDCG@10
TF-IDF      0.075000   0.689222  0.533814
BM25        0.084333   0.767000  0.632472

Resumen numérico:
{
  "TF-IDF": {
    "Precision@10": 0.075,
    "Recall@10": 0.6892,
    "NDCG@10": 0.5338
  },
  "BM25": {
    "Precision@10": 0.0843,
    "Recall@10": 0.767,
    "NDCG@10": 0.6325
  }
}


## 8. Comparación e interpretación de resultados

Resultados obtenidos en esta ejecución sobre las 300 consultas de test (promedios):

| Modelo | Precision@10 | Recall@10 | NDCG@10 |
|---|---|---|---|
| TF-IDF + coseno | 0.0750 | 0.6892 | 0.5338 |
| BM25 | 0.0843 | 0.7670 | 0.6325 |
| Diferencia (BM25 − TF-IDF) | +0.0093 | +0.0778 | +0.0987 |

**Por qué Precision@10 es baja y Recall@10 alta.** Los qrels de test de SciFact son binarios y muy dispersos: hay 339 juicios relevantes para 300 consultas, de modo que la mayoría de las consultas tiene **un único documento relevante**. Precision@10 divide los relevantes recuperados entre k = 10, aunque el modelo devuelva menos de diez resultados. Por eso, cuando una consulta tiene un único documento relevante y este aparece entre los diez primeros, **Precision@10 vale 1/10 = 0.10** y **Recall@10 vale 1/1 = 1.0**; si no aparece, ambas valen 0. Con esta estructura de qrels, Precision@10 apenas puede superar 0.10 en promedio, y el cociente Recall@10 / Precision@10 (≈ 9.2 en TF-IDF y ≈ 9.1 en BM25) es coherente con algo más de un documento relevante por consulta.

**Conclusión de esta ejecución:**
- **BM25 supera a TF-IDF en las tres métricas.** Recupera el documento relevante entre los diez primeros en más consultas (Recall@10 0.767 frente a 0.689, es decir, unas 7.8 puntos porcentuales más), lo que se traduce en una Precision@10 mayor (0.0843 frente a 0.0750).
- **La mayor ventaja está en NDCG@10** (0.6325 frente a 0.5338, +0.099, ≈ 18 % relativo), mayor que la mejora relativa de Recall@10 (≈ 11 %). Esto indica que BM25 no solo encuentra más documentos relevantes, sino que además los coloca en posiciones más altas del ranking.
- Las diferencias de Precision@10 parecen pequeñas en valor absoluto solo porque su techo práctico es ≈ 0.10; en relación con ese techo, la mejora de BM25 es del mismo orden que la de Recall@10.
- En este corpus, la saturación de la frecuencia de términos (k1) y la normalización por longitud del documento (b) de BM25 resultan más eficaces que la ponderación TF-IDF con similitud coseno para ordenar resúmenes científicos.

## 9. (Pendiente) Servicio web con FastAPI

En una siguiente fase se creará una API con dos endpoints:

- `GET /health`  
- `POST /search` con cuerpo `{"query": "...", "model": "tfidf|bm25", "top_k": 10}`

La API reutilizará las funciones `search_tfidf` y `search_bm25` definidas en este notebook, asegurando que los `document_id` devueltos sean los originales de SciFact.